# 07. The common hedge backtest

**The question.** What positions, trades, costs, and P&L does each hedging rule produce in the test period, when every rule runs through the same engine?

**Why a portfolio manager cares.** A comparison between hedges is only fair if they share the same dates, notional, execution timing, position limit, and cost treatment. This notebook produces the ledgers every later result is computed from.

**Learning objectives**

1. Work through the short Treasury leg's economics by hand.
2. Run every strategy through one engine at the base settings.
3. Reconcile the accounting: zero hedge, costs, aggregation, and timing.
4. Store a grid of rebalancing rules, delays, and financing spreads for notebook 08.

| | |
|---|---|
| Inputs | daily panel, `results/forecasts/forecasts_{test,dev}.csv`, freeze record |
| Outputs | `results/backtest/ledger_test_base_*.parquet`, `weekly_test_base.parquet`, `trades_test_base.csv`, `info_test_base.csv`, `weekly_test_grid.parquet`, development equivalents |
| Run after | 06 (full or smoke mode) |
| Approximate runtime | about 1 minute |

In [ ]:
# Setup: find the project root, load the configuration, and pick the run mode.
# The mode comes from MBSHEDGE_MODE (set by run_project.py) or config/project.yaml.
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config" / "project.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mbshedge.config import setup, experiment
from mbshedge import plotting as pt

cfg, paths, MODE = setup()
EXP = experiment(cfg, MODE)
pt.setup_style()
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

In [ ]:
from mbshedge import pipeline as pl
from mbshedge import backtest as bt
from mbshedge import freeze as fz
rec = fz.require_freeze(cfg, paths)
fz.log_test_evaluation(paths, rec, "07", "test-period backtests")
daily = pd.read_parquet(paths.processed / "daily_panel.parquet")
fc_test = pl.load_forecasts(paths.results / "forecasts" / "forecasts_test.csv")
fc_dev = pl.load_forecasts(paths.results / "forecasts" / "forecasts_dev.csv")
win = pl.windows(daily, EXP)
H = EXP["hedging"]; N = EXP["notional_usd"]
bdir = paths.results / "backtest"; bdir.mkdir(parents=True, exist_ok=True)
print(f"Freeze created {rec['created_utc']}. Test window: decisions {win['test']['first_decision']} to {win['test']['last_decision']}, "
      f"evaluation ends {win['test']['end_date'].date()}.")

## 1. The short leg, worked by hand

The portfolio holds $100mm of MBS exposure, kept constant (each day's gain or loss is swept to cash). Against it the manager sells $h$ dollars of the synthetic 10-year Treasury per dollar of MBS. A short Treasury position financed in repo (or a futures position) earns the bill rate on its collateral and pays the bond's total return, so its P&L is **minus h times the bond's excess return**:

$$
\text{P\&L}_t = N\left[\,r^{\text{MBS}}_t - r^f_t\,\right] \;-\; h_{t-1}\,N\left[\,r^{H}_t - r^f_t\,\right] \;-\; c\,N\,|\Delta h_t| ,
$$

where $h_{t-1}$ is the position held over day $t$ (set at the previous close) and $c$ is the one-way cost per dollar of hedge notional traded. All P&L is measured in excess of the bill return on capital.

A small example: the decision at the January 29 close says "forecast duration 4.0 years" and the hedge bond's duration is 8.0, so $h = 0.5$: sell $50mm of 10-year Treasuries. Execution is one trading day later (February 1 close). On February 2 the MBS excess return is +0.20% and the hedge bond's is +0.40%:

In [ ]:
toy = pd.DataFrame({"r_mbb_ex": [0.0, -0.004, 0.002, 0.001, -0.003, 0.001, 0.002, -0.001, 0.0005],
                    "r_h10_ex": [0.0, -0.008, 0.004, 0.002, -0.006, 0.0015, 0.003, -0.002, 0.001],
                    "dur_h10": 8.0, "days": [1, 3, 1, 1, 3, 1, 1, 1, 1], "dy10": 0.0},
                   index=pd.to_datetime(["2021-01-29", "2021-02-01", "2021-02-02", "2021-02-26", "2021-03-01",
                                         "2021-03-02", "2021-03-31", "2021-04-01", "2021-04-02"]))
dec = pd.DataFrame({"decision_month": pd.PeriodIndex(["2021-01", "2021-02", "2021-03"], freq="M"),
                    "decision_date": pd.to_datetime(["2021-01-29", "2021-02-26", "2021-03-31"]), "forecast": [4.0, 6.0, 6.0]})
toy_res = bt.run_backtest(toy, dec, end_date="2021-04-02", delay=1, rule="monthly", cost_bp=2.0)
L = toy_res["ledger"].copy()
by_hand = N * (0.002 - 0.5 * 0.004) - 2e-4 * N * 0.5
print(f"By hand, 2021-02-02: MBS +${N*0.002:,.0f}, hedge -${0.5*N*0.004:,.0f}, establishment cost ${2e-4*N*0.5:,.0f} -> net ${by_hand:,.0f}")
print(f"Engine,  2021-02-02: net ${N * L.loc['2021-02-02', 'net_pnl']:,.0f}")
assert abs(by_hand - N * L.loc["2021-02-02", "net_pnl"]) < 1e-6
(L * N).assign(h_held=L["h_held"], trade=L["trade"]).round(0)

The hedge gained less than the MBS on a day yields fell (both rallied), because the position was sized to duration 4 and the hedge bond's own duration is 8: a $50mm short of an 8-year-duration bond offsets the rate exposure of $100mm at duration 4. What remains is the mortgage basis: spread moves, prepayment surprises, and the part of the exposure the forecast missed.

## 2. Every strategy at the base settings (test period)

In [ ]:
strategies = ["UNH"] + [m for m in EXP["models"]["list"]]
res = pl.run_strategies(daily, fc_test, strategies, win["test"]["end_date"], delay=H["base_delay_days"], rule=H["base_rule"],
                        cost_bp=H["base_cost_bp"], financing_bp=H["financing_spread_bp_per_year"], max_h=H["max_hedge_ratio"])
pl.save_runs(res, bdir, "test_base")
info = pd.DataFrame({s: r["info"] for s, r in res.items()}).T
info[["start", "end", "years", "n_decisions", "n_trades", "turnover_per_year", "cost_per_year", "position_limit_hits"]]

## 3. Reconciliation

Each check compares two independent routes to the same number.

In [ ]:
checks = {}
u = res["UNH"]["ledger"]
checks["zero hedge reproduces the unhedged MBS excess return"] = bool(np.allclose(u["net_pnl"], u["mbs_pnl"]) and u["cost"].sum() == 0)
c = H["base_cost_bp"] / 1e4
checks["cost = c x total |trade| for every strategy"] = all(abs(r["ledger"]["cost"].sum() - c * r["ledger"]["trade"].abs().sum()) < 1e-12 for r in res.values())
checks["weekly gross P&L sums to daily gross P&L (within the evaluated weeks)"] = all(
    abs(r["weekly"]["gross_pnl"].sum() - r["ledger"]["gross_pnl"].groupby(pd.Grouper(freq="W-FRI")).sum()
        .reindex(r["weekly"].index).sum()) < 1e-10 for r in res.values())
checks["weekly costs add up to every ledger cost, including establishment and liquidation"] = all(
    abs(r["weekly"]["cost"].sum() - r["ledger"]["cost"].sum()) < 1e-12 for r in res.values())
pos_ok = True
for s, r in res.items():
    t = r["trades"]; executed = set(pd.to_datetime(t.loc[t["executed"], "execution_date"]))
    h_ = r["ledger"]["h_held"]; changed = h_.index[1:][np.abs(np.diff(h_.to_numpy())) > 0]
    prev_days = {daily.index[daily.index.get_loc(d) - 1] for d in changed}
    pos_ok &= prev_days <= executed
checks["positions change only on the day after an executed trade"] = bool(pos_ok)
checks["all strategies share the same evaluation weeks"] = len({tuple(r["weekly"].index) for r in res.values()}) == 1
for k, v in checks.items():
    print(("PASS " if v else "FAIL ") + k)
assert all(checks.values())

## 4. Hedged duration through time

The hedged duration is the hedge ratio times the hedge bond's duration, the exposure in years each strategy offsets. The static line moves only because the hedge bond's duration changes with the yield level.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for s in ["S", "U", "E", "H", "X"]:
    L = res[s]["ledger"]
    hd = L["h_held"] * daily.loc[L.index, "dur_h10"].shift(1).bfill()
    st = pt.style(s)
    ax.plot(hd.index, hd, color=st["color"], ls=st["linestyle"], lw=1.6 if s in ("S", "U", "H") else 1.1, label=st["label"], drawstyle="steps-post")
pt.shade_episodes(ax, [e for e in EXP["episodes"] if e["period"] == "test"], end_default=daily.index.max())
ax.set_ylabel("years"); ax.set_title("Duration hedged by each rule (test period)"); ax.legend(ncol=2, fontsize=8.5)
pt.save_figure(fig, "07_hedged_duration_test", paths, mode=MODE)
plt.show()

## 5. The grid for notebook 08: rules, delays, and financing

Every strategy is rerun under each rebalancing rule (monthly, quarterly, and duration bands of 0.10, 0.25, and 0.50 years), each execution delay (0, 1, and 5 trading days), and with a 10 bp per year financing spread on the short. Costs are stored at 1 bp so any cost level can be applied later without rerunning: net P&L at cost $c$ equals gross P&L minus $c$ times the 1 bp cost.

In [ ]:
grid = pl.grid_runs(daily, fc_test, strategies, win["test"]["end_date"], EXP, H["rebalance_rules"], H["execution_delay_days"],
                    [H["financing_spread_sensitivity_bp"]])
grid.to_parquet(bdir / "weekly_test_grid.parquet")
print(f"Grid: {grid.groupby(['strategy', 'rule', 'delay', 'financing_bp']).ngroups} configurations, {len(grid)} rows")

## 6. The development period, for reference

The same engine on development decisions (April 2010 to December 2014). The static hedge is in-sample here (its duration is the development average), and the economic and hybrid hedges use the retrospective proxy before mid-2013, so these numbers describe development behavior only.

In [ ]:
res_dev = pl.run_strategies(daily, fc_dev, strategies, win["dev"]["end_date"], delay=H["base_delay_days"], rule=H["base_rule"],
                            cost_bp=H["base_cost_bp"], financing_bp=H["financing_spread_bp_per_year"], max_h=H["max_hedge_ratio"])
pl.save_runs(res_dev, bdir, "dev_base")
from mbshedge import evaluation as ev
dev_sum = ev.strategy_summary({s: r["weekly"] for s, r in res_dev.items()}, {s: r["info"] for s, r in res_dev.items()}, N)
dev_sum.to_csv(bdir / "summary_dev_base.csv")
dev_sum[["vol_gross_ann_pct", "var_reduction_vs_unhedged", "residual_duration_years", "turnover_per_year", "cost_drag_bp_per_year"]].round(3)